<a href="https://colab.research.google.com/github/bosglas-source/NLP-Fraud-Detection-project/blob/main/02_features_embeddings_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02 - Features, Embeddings, and Similarity

This notebook starts from the cleaned file produced by Notebook 01:

`contracts_ie_clean.csv`

It creates reusable text features for the modeling notebook:

- TF-IDF matrix and fitted vectorizer
- Multilingual sentence embeddings
- Cross-buyer cosine similarity scores for copy-paste detection
- A feature CSV with `copy_paste_description`

Designed to run in Google Colab temporary storage or local Jupyter. In Colab, upload `contracts_ie_clean.csv` when prompted.

## 0. Setup

In [4]:
from pathlib import Path
import sys
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Running in:', 'Google Colab' if IN_COLAB else 'Local Jupyter')

if IN_COLAB:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'sentence-transformers', 'scikit-learn', 'scipy', 'joblib', 'tqdm'
    ], check=True)

DATA_DIR = Path.cwd()
OUTPUT_DIR = DATA_DIR / 'outputs_02'
OUTPUT_DIR.mkdir(exist_ok=True)

CSV_PATH = DATA_DIR / 'contracts_ie_clean.csv'
print('DATA_DIR  :', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

Running in: Google Colab
DATA_DIR  : /content
OUTPUT_DIR: /content/outputs_02


In [5]:
# Colab convenience: upload the cleaned CSV directly into temporary storage.
if IN_COLAB and not CSV_PATH.exists():
    from google.colab import files
    print('Upload contracts_ie_clean.csv')
    uploaded = files.upload()
    if 'contracts_ie_clean.csv' not in uploaded:
        raise FileNotFoundError('Please upload a file named contracts_ie_clean.csv')

if not CSV_PATH.exists():
    raise FileNotFoundError(f'Could not find {CSV_PATH}. Put contracts_ie_clean.csv next to this notebook.')

## 1. Load and Validate Data

In [6]:
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

df = pd.read_csv(CSV_PATH, low_memory=False)
print(f'Loaded {len(df):,} rows and {df.shape[1]} columns')

required_cols = [
    'contract_id', 'title', 'description', 'buyer_id', 'buyer_name',
    'single_bid', 'short_tender_period', 'winner_concentration'
]
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f'Missing required columns from Notebook 01 output: {missing}')

df[required_cols].head(3)

Loaded 139,629 rows and 28 columns


,contract_id,title,description,buyer_id,buyer_name,single_bid,short_tender_period,winner_concentration
0,IE_be309cda596edfda243ef05e7b714ad0b6ede41e879...,Redesign of the Housing Finance Agency plc. we...,Housing Finance Agency plc. (HFA) invites tend...,IE_body_28a3297134ffd186fe6814e1da1ae1491ad2bf...,Housing Finance Agency Plc,NaN,NaN,0
1,IE_71396d4025cbdee0da629e56a91bf26109feda49171...,Discover Primary Science & Greenwave Appointme...,Discover Primary Science wishes to contract th...,IE_body_68f9955a5f5ab6dcd5ed8895517a9015c4199a...,IDA Ireland,NaN,1.0,0
2,IE_dbd22e43d9f750985c0c5cd05e0f790d9acd38c8018...,Provision of Security Services to IDA Ireland,Provision of Security Services to IDA Ireland,IE_body_68f9955a5f5ab6dcd5ed8895517a9015c4199a...,IDA Ireland,NaN,1.0,0


## 2. Build Text Field

In [7]:
df['text_for_model'] = df['description'].fillna('').astype(str).str.strip()
empty_description = df['text_for_model'].eq('')
df.loc[empty_description, 'text_for_model'] = df.loc[empty_description, 'title'].fillna('').astype(str).str.strip()

texts = df['text_for_model'].tolist()

print(f'Text rows              : {len(texts):,}')
print(f'Empty after title fill : {sum(t == "" for t in texts):,}')
print('\nSample text:')
print(texts[0][:500])

Text rows              : 139,629
Empty after title fill : 1

Sample text:
Housing Finance Agency plc. (HFA) invites tenders from interested parties who wish to put forward proposals relating to the redesigning of its website (www.hfa.ie). The HFA wishes to draw the attention of interested parties to the contents of this document and in particular to the sections entitled, ‘Supplier Response Format’, ‘Evaluation of Tenders and Award Criteria’ and ‘Instructions to Tenderers’. The HFA wishes to deal with a single supplier. In the event of a group of tenderers jointly sub


## 3. TF-IDF Baseline

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz
import joblib
import time

tfidf = TfidfVectorizer(
    sublinear_tf=True,
    max_features=50_000,
    ngram_range=(1, 2),
    min_df=3,
    strip_accents='unicode',
    analyzer='word'
)

t0 = time.time()
X_tfidf = tfidf.fit_transform(texts)
elapsed = time.time() - t0

print(f'TF-IDF shape       : {X_tfidf.shape}')
print(f'Non-zero values    : {X_tfidf.nnz:,}')
print(f'Sparsity           : {100 * (1 - X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])):.2f}%')
print(f'Completed in       : {elapsed:.1f} seconds')

save_npz(OUTPUT_DIR / 'tfidf_matrix.npz', X_tfidf)
joblib.dump(tfidf, OUTPUT_DIR / 'tfidf_vectorizer.joblib')
print('Saved TF-IDF artifacts')

TF-IDF shape       : (139629, 50000)
Non-zero values    : 12,722,266
Sparsity           : 99.82%
Completed in       : 39.4 seconds
Saved TF-IDF artifacts


In [9]:
feature_names = np.array(tfidf.get_feature_names_out())
mean_scores = np.asarray(X_tfidf.mean(axis=0)).ravel()
top_idx = mean_scores.argsort()[-20:][::-1]

pd.DataFrame({
    'term': feature_names[top_idx],
    'mean_tfidf': mean_scores[top_idx]
})

,term,mean_tfidf
0,the,0.039023
1,of,0.035136
2,and,0.033433
3,to,0.031099
4,for,0.027166
5,in,0.020308
6,at,0.017975
7,this,0.017651
8,services,0.017621
9,be,0.017012


## 4. Sentence Embeddings

In [10]:
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Set SAMPLE_SIZE to a smaller number for a quick CPU test, or None for the full dataset.
SAMPLE_SIZE = None

if SAMPLE_SIZE is None:
    encode_df = df.copy()
else:
    encode_df = df.head(SAMPLE_SIZE).copy()

encode_texts = encode_df['text_for_model'].tolist()
print(f'Device         : {DEVICE}')
print(f'Rows to encode : {len(encode_texts):,}')

encoder = SentenceTransformer(MODEL_NAME, device=DEVICE)
print('Embedding dim  :', encoder.get_sentence_embedding_dimension())

Device         : cuda
Rows to encode : 139,629


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dim  : 768


In [11]:
batch_size = 128 if DEVICE == 'cuda' else 32

t0 = time.time()
embeddings = encoder.encode(
    encode_texts,
    batch_size=batch_size,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype('float32')
elapsed = time.time() - t0

print(f'Embeddings shape : {embeddings.shape}')
print(f'Dtype            : {embeddings.dtype}')
print(f'Completed in     : {elapsed / 60:.1f} minutes')

np.save(OUTPUT_DIR / 'embeddings.npy', embeddings)
encode_df[['contract_id']].to_csv(OUTPUT_DIR / 'contract_ids.csv', index=False)
print('Saved embeddings.npy and contract_ids.csv')

Batches:   0%|          | 0/1091 [00:00<?, ?it/s]

Embeddings shape : (139629, 768)
Dtype            : float32
Completed in     : 13.8 minutes
Saved embeddings.npy and contract_ids.csv


In [12]:
norms = np.linalg.norm(embeddings[:min(1000, len(embeddings))], axis=1)
print(f'Norm check: min={norms.min():.6f}, max={norms.max():.6f}, mean={norms.mean():.6f}')

Norm check: min=1.000000, max=1.000000, mean=1.000000


## 5. Cross-Buyer Similarity Scan

In [13]:
from tqdm.auto import tqdm

SIMILARITY_THRESHOLD = 0.92
BLOCK_SIZE = 2000

n = len(embeddings)
buyer_ids = encode_df['buyer_id'].fillna('__missing_buyer__').astype(str).values
contract_ids = encode_df['contract_id'].astype(str).values

copy_paste_flags = np.zeros(n, dtype=np.int8)
flagged_pairs = []

print(f'Scanning {n:,} contracts in {BLOCK_SIZE}-row blocks')
print(f'Threshold: cosine similarity >= {SIMILARITY_THRESHOLD}')

t0 = time.time()
for i in tqdm(range(0, n, BLOCK_SIZE)):
    block_a = embeddings[i:i + BLOCK_SIZE]
    buyers_a = buyer_ids[i:i + BLOCK_SIZE]

    for j in range(i, n, BLOCK_SIZE):
        block_b = embeddings[j:j + BLOCK_SIZE]
        buyers_b = buyer_ids[j:j + BLOCK_SIZE]

        sim = block_a @ block_b.T
        if i == j:
            np.fill_diagonal(sim, 0.0)

        different_buyer = buyers_a[:, None] != buyers_b[None, :]
        rows, cols = np.where((sim >= SIMILARITY_THRESHOLD) & different_buyer)

        for r, c in zip(rows, cols):
            a = i + r
            b = j + c
            if a >= b:
                continue
            copy_paste_flags[a] = 1
            copy_paste_flags[b] = 1
            flagged_pairs.append({
                'contract_id_a': contract_ids[a],
                'contract_id_b': contract_ids[b],
                'buyer_id_a': buyer_ids[a],
                'buyer_id_b': buyer_ids[b],
                'cosine_sim': round(float(sim[r, c]), 5)
            })

elapsed = time.time() - t0
print(f'Finished in {elapsed / 60:.1f} minutes')
print(f'Flagged pairs            : {len(flagged_pairs):,}')
print(f'Unique contracts flagged : {copy_paste_flags.sum():,}')

Scanning 139,629 contracts in 2000-row blocks
Threshold: cosine similarity >= 0.92


  0%|          | 0/70 [00:00<?, ?it/s]

Finished in 9.4 minutes
Flagged pairs            : 59,374
Unique contracts flagged : 31,584


In [14]:
similarity_df = pd.DataFrame(flagged_pairs)

if not similarity_df.empty:
    id_to_buyer_name = dict(zip(encode_df['contract_id'].astype(str), encode_df['buyer_name'].fillna('').astype(str)))
    id_to_title = dict(zip(encode_df['contract_id'].astype(str), encode_df['title'].fillna('').astype(str)))

    similarity_df = similarity_df.sort_values('cosine_sim', ascending=False).reset_index(drop=True)
    similarity_df['buyer_name_a'] = similarity_df['contract_id_a'].map(id_to_buyer_name)
    similarity_df['buyer_name_b'] = similarity_df['contract_id_b'].map(id_to_buyer_name)
    similarity_df['title_a'] = similarity_df['contract_id_a'].map(id_to_title)
    similarity_df['title_b'] = similarity_df['contract_id_b'].map(id_to_title)

similarity_df.to_csv(OUTPUT_DIR / 'similarity_scores.csv', index=False)
print(f'Saved {len(similarity_df):,} rows to similarity_scores.csv')
similarity_df.head(10)

Saved 59,374 rows to similarity_scores.csv


,contract_id_a,contract_id_b,buyer_id_a,buyer_id_b,cosine_sim,buyer_name_a,buyer_name_b,title_a,title_b
0,IE_1c419348afaa8f154080a4617c4377ae25837e1f589...,IE_1774467944be960972def44cd5571253b925753bd55...,IE_body_7dd9dc1705967f9b605651d1ef88ae57058a67...,IE_body_3aee3a4ee3a691892bc73ea124a2e9afdba810...,1.0,Safefood,An Post National Lottery Company,Market Research Services,Market Research Services
1,EU_020dd3561675f660cd7edf28859c63ef16c349380a4...,EU_edf388f4b565bdc8e08d3148fb3391917ddcf8d58ed...,EU_body_58debb53c15394a7ba2e621e633a869956f0ed...,EU_body_c60d376676fb25912b1ec931e244afdf1fada3...,1.0,Irish Prison Service,Lackagh National School,Proban Treated Fabric,Civil Structural Engineering Services for Addi...
2,EU_020dd3561675f660cd7edf28859c63ef16c349380a4...,EU_4bfc92d478891dac2fd7283830eb9fd3ad11173b290...,EU_body_58debb53c15394a7ba2e621e633a869956f0ed...,EU_body_691fa4621731a90aff4935a82c6e172cfad0b2...,1.0,Irish Prison Service,Bord Bia (Irish Food Board),Proban Treated Fabric,Creative Agency for B2C Beef Campaign and Mana...
3,EU_fd01d8584083c9fc1cc0c0e8b70e07b024fde3c5bc2...,EU_4bfc92d478891dac2fd7283830eb9fd3ad11173b290...,EU_body_1e3741b06634f423fe3ca13d7813cde082c784...,EU_body_691fa4621731a90aff4935a82c6e172cfad0b2...,1.0,Office of Public Works (OPW),Bord Bia (Irish Food Board),Tractors in 4 Lots,Creative Agency for B2C Beef Campaign and Mana...
4,EU_fd01d8584083c9fc1cc0c0e8b70e07b024fde3c5bc2...,EU_468d780b3efacb20fdef153a3efdaffddc09e6a49c6...,EU_body_1e3741b06634f423fe3ca13d7813cde082c784...,EU_body_6137410da3e8fb5cb1a84d11a4e9e1f39dd38d...,1.0,Office of Public Works (OPW),Department of Rural and Community Development,Tractors in 4 Lots,West Cork Islands Heavy Cargo Service 2021-2024
5,EU_fd01d8584083c9fc1cc0c0e8b70e07b024fde3c5bc2...,EU_b3f938c8f92945eaf82c4ca079b1d3aa84cc5a8c46d...,EU_body_1e3741b06634f423fe3ca13d7813cde082c784...,EU_body_6137410da3e8fb5cb1a84d11a4e9e1f39dd38d...,1.0,Office of Public Works (OPW),Department of Rural and Community Development,Tractors in 4 Lots,RFT Passenger Ferry Service to Sherkin Island ...
6,EU_fd01d8584083c9fc1cc0c0e8b70e07b024fde3c5bc2...,EU_8c0a4a878f9ac0838761d80e9f8341e01cc77af26c3...,EU_body_1e3741b06634f423fe3ca13d7813cde082c784...,EU_body_4157efcd4ab56ca4c08e8d44e3529dbaff5c34...,1.0,Office of Public Works (OPW),Our Lady of the Wayside National School (Blueb...,Tractors in 4 Lots,Civil Structural Engineering Services for Addi...
7,EU_fd01d8584083c9fc1cc0c0e8b70e07b024fde3c5bc2...,EU_06e426f422863f8a181f0832cbfa8736fb27ff2045d...,EU_body_1e3741b06634f423fe3ca13d7813cde082c784...,EU_body_e01a745f1d1b64ed4a077b5efaf4d2994bc30d...,1.0,Office of Public Works (OPW),Health Service Executive (HSE),Tractors in 4 Lots,Extension to RFT 177177 — 15318 PIN for Ultras...
8,EU_fd01d8584083c9fc1cc0c0e8b70e07b024fde3c5bc2...,EU_a55770702c89bc7587cafa9d74d28f9f5c1ae530952...,EU_body_1e3741b06634f423fe3ca13d7813cde082c784...,EU_body_af4bb07550903f245c2340b31a1bb0506cd818...,1.0,Office of Public Works (OPW),Galway Roscommon Education & Training Board,Tractors in 4 Lots,Managed ICT Services for GRETB
9,EU_020dd3561675f660cd7edf28859c63ef16c349380a4...,IE_08e5d89c1af170b0fa1aa14d15a15103b8631e386a3...,EU_body_58debb53c15394a7ba2e621e633a869956f0ed...,IE_body_0055f0922581881226358f902165fd2a63f248...,1.0,Irish Prison Service,Cork County Council,Proban Treated Fabric,Bandon Shelving and Furniture - Fit out of Ban...


## 6. Save Feature File for Notebook 03

In [15]:
features_df = encode_df.copy()
features_df['copy_paste_description'] = copy_paste_flags

feature_path = OUTPUT_DIR / 'contracts_ie_features.csv'
features_df.to_csv(feature_path, index=False)

print(f'Saved feature file: {feature_path}')
print(f'Rows: {len(features_df):,}')
print(f'copy_paste_description rate: {features_df["copy_paste_description"].mean() * 100:.2f}%')

Saved feature file: /content/outputs_02/contracts_ie_features.csv
Rows: 139,629
copy_paste_description rate: 22.62%


In [16]:
deliverables = [
    'tfidf_matrix.npz',
    'tfidf_vectorizer.joblib',
    'embeddings.npy',
    'contract_ids.csv',
    'similarity_scores.csv',
    'contracts_ie_features.csv'
]

print('Notebook 02 deliverables')
print('-' * 60)
for name in deliverables:
    path = OUTPUT_DIR / name
    size_mb = path.stat().st_size / (1024 * 1024) if path.exists() else 0
    status = 'OK' if path.exists() else 'MISSING'
    print(f'{name:<30} {status:<8} {size_mb:>8.2f} MB')

Notebook 02 deliverables
------------------------------------------------------------
tfidf_matrix.npz               OK         113.09 MB
tfidf_vectorizer.joblib        OK           1.96 MB
embeddings.npy                 OK         409.07 MB
contract_ids.csv               OK           9.32 MB
similarity_scores.csv          OK          29.41 MB
contracts_ie_features.csv      OK         269.58 MB
